# Read and write in own words

### Why use the "multimodal only" dataset for a text-only project?



* **It filters out massive automated platform noise:** The "all text" dumps of Fakeddit include millions of blank posts, auto-generated bot threads, spam links, and deleted titles. The authors created the `multimodal_only` subset by isolating posts where a human user uploaded an image *and* wrote an original textual title. This process acts as a high-quality filter, leaving you with genuine, human-written content.
* **It preserves benchmark parity:** In the official Fakeddit paper, the authors explicitly state that their baseline experiments and error analyses were performed *strictly* on the multimodal samples. If you want to compare your BERT model's accuracy against the official published academic standard, you must train on the exact same data split they used.
* **Linguistic Richness:** Titles attached to images on Reddit (like memes, sensationalized news screenshots, or altered photos) are inherently richer in the specific types of manipulation you want to study—such as sarcasm, irony, misleading claims, and baiting language.

---

### Why not load test data now? 

Loading test data in exploratory phase would present a risk of data leakage.

* **Preventing Cognitive Bias:** If you compute class balances, check descriptive statistics, or look at vocabulary distributions on your test data, you might unconsciously write preprocessing rules or design choices tailored to those specific test samples. This compromises your model's validity.
* **Preserving the "Golden Standard":** Your test dataset must remain a completely unseen environment. Its only job is to sit quietly until the very end of your project. Once your model is fully trained on the `train` set and tuned using the `validate` set, you will pass the `test` data through it *once* to get your final thesis numbers.
* **Evaluation Strategy:** Eventually need to hand-label a tiny, custom *UK* test set to evaluate final system, the US Fakeddit test set is less critical for your immediate daily workflow. Your current focus is purely on building a working training pipeline.

---


1. **Train Set:** Used to let the BERT model learn patterns.
2. **Validation Set:** Used to check performance during training and tweak hyperparameters.
3. **Test Set:** untouched until evaluation


### 1.1 Read files

In [1]:
import pandas as pd
import os

data_dir = os.path.join('..', 'data', 'processed', 'US')

train_path = os.path.join(data_dir, 'multimodal_train.tsv')
val_path = os.path.join(data_dir, 'multimodal_validate.tsv')

train_fakeddit = pd.read_csv(train_path, sep='\t')
val_fakeddit = pd.read_csv(val_path, sep='\t')

In [2]:
train_fakeddit.head()

,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,1.551641e+09,i.imgur.com,True,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,NaN,2.0,12,mildlyinteresting,My Walgreens offbrand Mucinex was engraved wit...,0.84,1,0,0
1,VIDCAs17,this concerned sink with a tiny hat,1.534727e+09,i.redd.it,True,98pbid,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,NaN,2.0,119,pareidolia,This concerned sink with a tiny hat,0.99,0,2,2
2,prometheus1123,hackers leak emails from uae ambassador to us,1.496511e+09,aljazeera.com,True,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,NaN,1.0,44,neutralnews,Hackers leak emails from UAE ambassador to US,0.92,1,0,0
3,NaN,puppy taking in the view,1.471341e+09,i.imgur.com,True,4xypkv,https://external-preview.redd.it/HLtVNhTR6wtYt...,NaN,26.0,250,photoshopbattles,PsBattle: Puppy taking in the view,0.95,1,0,0
4,3rikR3ith,i found a face in my sheet music too,1.525318e+09,i.redd.it,True,8gnet9,https://preview.redd.it/ri7ut2wn8kv01.jpg?widt...,NaN,2.0,13,pareidolia,I found a face in my sheet music too!,0.84,0,2,2


### 1.2. Isolate relevant columns

In [16]:
rel_cols = ['clean_title', 'subreddit', 'domain', 'score', 'num_comments', 'upvote_ratio', 'created_utc', '2_way_label', '3_way_label', '6_way_label']
train_df = train_fakeddit[rel_cols].copy()
val_df = val_fakeddit[rel_cols].copy()

dfs = {'train': train_df, 'val': val_df}

In [4]:
# Check for NaNs
print(train_fakeddit.isna().sum())

author                   28710
clean_title                  0
created_utc                  0
domain                  167857
hasImage                     0
id                           0
image_url                 1534
linked_submission_id    396143
num_comments            167857
score                        0
subreddit                    0
title                        0
upvote_ratio            167857
2_way_label                  0
3_way_label                  0
6_way_label                  0
dtype: int64


In [5]:
train_df.dtypes

clean_title         str
subreddit           str
domain              str
score             int64
num_comments    float64
upvote_ratio    float64
created_utc     float64
2_way_label       int64
3_way_label       int64
6_way_label       int64
dtype: object

Fill NaN even if none present to avoid downstream issues

In [6]:
def fill_na_values(df):
    for col in df.columns: 
        if df[col].dtype == str:
            df[col] = df[col].fillna('')
        elif df[col].dtype == int:
            df[col] = df[col].fillna(0).astype(int)
        elif df[col].dtype == float:
            df[col] = df[col].fillna(0).astype(float)
        else:
            df[col] = df[col].fillna('')
    return df

train_df = fill_na_values(train_df)
val_df = fill_na_values(val_df)

### 1.3 Map labels

In [32]:
fakeddit_3_way_labels = {
    0: "True",
    1: "Fake/Misleading",
    2: "Satire",
}

train_df['label_name'] = train_df['3_way_label'].map(fakeddit_3_way_labels)
val_df['label_name'] = val_df['3_way_label'].map(fakeddit_3_way_labels)

Sample titles per class to make sure label mapping is correct

In [35]:
for label in train_df['label_name'].unique():
    print(f"5 post tiles for label: {label}")
    display(train_df.loc[train_df['label_name'] == label]['clean_title'].head())

5 post tiles for label: True


0    my walgreens offbrand mucinex was engraved wit...
2        hackers leak emails from uae ambassador to us
3                             puppy taking in the view
5    bride and groom exchange vows after fatal shoo...
7    rabbi meat from cloned pig could be kosher for...
Name: clean_title, dtype: str

5 post tiles for label: Satire


1                  this concerned sink with a tiny hat
4                 i found a face in my sheet music too
6                                        major thermos
8                                              cutouts
9    jesus christ converting local teens to christi...
Name: clean_title, dtype: str

5 post tiles for label: Fake/Misleading


10      victory the great european crusade vichy france
18       love and peace the washington way soviet union
24    applying to join the chinese communist party m...
64    he cant do it alone proempire long long ago da...
65    the flying tigers of the free world strike aga...
Name: clean_title, dtype: str

In [8]:
train_df.head()

,clean_title,subreddit,domain,score,num_comments,upvote_ratio,created_utc,2_way_label,3_way_label,6_way_label,label_name
0,my walgreens offbrand mucinex was engraved wit...,mildlyinteresting,i.imgur.com,12,2.0,0.84,1.551641e+09,1,0,0,True
1,this concerned sink with a tiny hat,pareidolia,i.redd.it,119,2.0,0.99,1.534727e+09,0,2,2,Satire
2,hackers leak emails from uae ambassador to us,neutralnews,aljazeera.com,44,1.0,0.92,1.496511e+09,1,0,0,True
3,puppy taking in the view,photoshopbattles,i.imgur.com,250,26.0,0.95,1.471341e+09,1,0,0,True
4,i found a face in my sheet music too,pareidolia,i.redd.it,13,2.0,0.84,1.525318e+09,0,2,2,Satire


### 2.1 Examine virality metrics

See how upvotes and comments vary across classes



In [9]:
virality_summary = train_df.groupby('label_name')[['score', 'num_comments', 'upvote_ratio']].agg(['mean', 'median', 'max'])

virality_summary

score                num_comments                  \
                       mean median     max         mean median      max   
label_name                                                                
Fake/Misleading  256.562536   65.0   18724    18.288778    6.0   1906.0   
Satire           225.762021   12.0   93294     3.670538    0.0   8832.0   
True             654.781255   15.0  137179    29.761249    3.0  10783.0   

                upvote_ratio              
                        mean median  max  
label_name                                
Fake/Misleading     0.932473   0.96  1.0  
Satire              0.430324   0.00  1.0  
True                0.831929   0.85  1.0

### 2.2 Subreddit distribution

In [11]:
print("Top 10 Subreddits in Dataset")
print(train_df['subreddit'].value_counts().head(10))

# See how subreddits cross-reference with labels
print("\n--- Example: Subreddits making up 'Manipulated Content' (Label 4) ---")
print(train_df[train_df['6_way_label'] == 4]['subreddit'].value_counts().head(5))

Top 10 Subreddits in Dataset
subreddit
psbattle_artwork        167857
mildlyinteresting        86237
photoshopbattles         55198
pareidolia               47331
fakehistoryporn          35576
nottheonion              31977
upliftingnews            23960
fakealbumcovers          21725
misleadingthumbnails     15962
propagandaposters        13848
Name: count, dtype: int64

--- Example: Subreddits making up 'Manipulated Content' (Label 4) ---
subreddit
psbattle_artwork    167857
Name: count, dtype: int64


### 2.3 Class Balance per split

In [ ]:
for split,df in dfs.items():
    print(split.upper())
    print(df['label_name'].value_counts()/len(df))
    print('\n')

TRAIN
label_name
Satire             0.581686
True               0.393761
Fake/Misleading    0.024553
Name: count, dtype: float64


VAL
label_name
Satire             0.584021
True               0.392976
Fake/Misleading    0.023002
Name: count, dtype: float64




### Title length distribution. 

Informs BERT max sequence length

Majority well under 32 words - will probably use 32 tokens

In [38]:
title_word_counts = train_df['clean_title'].apply(lambda x: len(x.split()))
print(title_word_counts.describe())
print(f'Number of posts with titles longer than 32 words: {title_word_counts[title_word_counts > 32].count()} out of a total of {len(train_df)} posts')

count    564000.000000
mean          7.459456
std           5.639149
min           1.000000
25%           3.000000
50%           6.000000
75%          10.000000
max         553.000000
Name: clean_title, dtype: float64
Number of posts with titles longer than 32 words: 3021 out of a total of 564000 posts
